In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm


## Optimal transient growth in a spatially-developing flow

So far we have optimised abstract benchmark functions. Here we optimise something with a genuine fluid-mechanics flavour: a small disturbance $q(x,t)$ convected through a 1D flow that is locally unstable only in a bounded region of space (e.g. a "pocket" of a boundary layer, jet or wake). This is the classical **(non-parallel) linear Ginzburg-Landau model**, used throughout hydrodynamic stability and flow-control theory as a tractable stand-in for the linearised Navier-Stokes equations:

- C. Cossu, *"An Introduction to Optimal Control"*, lecture notes from the FLOW-NORDITA Summer School 2013 (in `references/`), §6.1 "An Example: Optimal Growth for the Nonparallel Ginzburg-Landau Model" — this notebook implements the exercise stated there.
- S. Bagheri, D.S. Henningson, J. Hœpffner, P.J. Schmid, *"Input-Output Analysis and Control Design Applied to a Linear Model of Spatially Developing Flows"*, Appl. Mech. Rev. 62 (2009) — they use the same equation to distinguish **oscillators** (globally unstable flows) from **noise amplifiers** (globally stable, but capable of huge *transient* energy amplification of a well-chosen disturbance). We are building the noise-amplifier case.

The governing PDE is

$$
\frac{\partial q}{\partial t} = \sigma(x)\, q - U \frac{\partial q}{\partial x} + \mu \frac{\partial^2 q}{\partial x^2}, \qquad x \in [-X, X], \quad q(\pm X, t) = 0,
$$

with a spatially-varying growth rate

$$
\sigma(x) = \sigma_0 - \sigma_2 \frac{x^2}{2}, \qquad \sigma_2 > 0,
$$

which is locally *unstable* ($\sigma(x) > 0$) only in a pocket around $x=0$ and stable everywhere else. $U$ is the convection speed and $\mu$ the diffusion (damping) coefficient.

Unlike Cossu's own example code, which advances the discretized system exactly using the matrix exponential (`expm(L*T)`), you will write your **own forward time-stepper** for the discretized state equation (and its adjoint), and use the exact matrix-exponential solution only as an independent check on your results — never as the solver itself.

In [ ]:
def build_operator(Nx, X, U, mu, sigma0, sigma2):
    """Discretize the Ginzburg-Landau operator L such that dq/dt = L @ q.

    Nx interior grid intervals -> N = Nx-1 unknowns (boundary values are 0
    and not part of the state vector).
    """
    dx = 2*X / Nx
    N = Nx - 1
    j = np.arange(1, Nx)
    x = -X + 2*X*j/Nx

    sigma =                          #>>> sigma(x) = sigma0 - sigma2 * x**2 / 2

    D1 = np.zeros((N, N))   # centered first derivative  d/dx
    D2 = np.zeros((N, N))   # centered second derivative  d^2/dx^2
    for k in range(N):
        if k - 1 >= 0:
            D1[k, k-1] = -1/(2*dx)
            D2[k, k-1] = 1/dx**2
        if k + 1 < N:
            D1[k, k+1] = 1/(2*dx)
            D2[k, k+1] = 1/dx**2
        D2[k, k] = -2/dx**2

    L = np.diag(sigma) - U*D1 + mu*D2
    return L, x, dx


In [ ]:
U, mu, X, sigma2, sigma0 = 2.0, 1.0, 20.0, 0.1, 0.8
Nx = 160

L, x, dx = build_operator(Nx, X, U, mu, sigma0, sigma2)
N = L.shape[0]
print('N =', N, ' dx =', dx)

sigma = sigma0 - sigma2*x**2/2
plt.figure(figsize=(6, 3))
plt.plot(x, sigma)
plt.axhline(0, color='k', lw=0.8)
plt.fill_between(x, 0, sigma, where=(sigma > 0), color='r', alpha=0.2, label='locally unstable pocket')
plt.xlabel('x'); plt.ylabel(r'$\sigma(x)$')
plt.legend()
plt.show()


Check that the discretization is sane before trusting anything built on top of it: plot the eigenvalues of $L$. Everything should lie in the left half-plane (the flow is **globally stable**, i.e. no eigenvalue grows), even though $\sigma(x)>0$ in the pocket — this is precisely the "noise amplifier" regime: locally unstable, globally stable.

In [ ]:
eigL = np.linalg.eigvals(L)
plt.figure(figsize=(5, 5))
plt.scatter(eigL.real, eigL.imag, s=15)
plt.axvline(0, color='r', lw=1)
plt.xlabel('Re'); plt.ylabel('Im')
plt.title('Spectrum of L')
plt.show()
print('largest real part of an eigenvalue:', eigL.real.max(), ' (should be < 0)')


## Part 1 — Forward time integration

We need a way to solve $\dot q = Lq$, $q(0)=q_0$ numerically. Implement the two simplest time-steppers below: explicit (forward) **Euler** and classical **4th-order Runge-Kutta (RK4)**. The generic driver `integrate` (given) calls whichever stepper you pass it, and works for both forward ($t_1 > t_0$) and backward ($t_1 < t_0$) integration — you will need the backward direction in Part 2.

In [ ]:
def euler_step(Mat, y, dt):
    #>>> one forward-Euler step for dy/dt = Mat @ y
    pass

def rk4_step(Mat, y, dt):
    #>>> one classical RK4 step for dy/dt = Mat @ y
    pass

def integrate(step_fn, Mat, y0, t0, t1, dt):
    "Advance dy/dt = Mat @ y from t0 to t1 (t1 may be less than t0) using step_fn."
    nsteps = int(round(abs(t1 - t0) / dt))
    h = dt if t1 > t0 else -dt
    y = y0.copy()
    for _ in range(nsteps):
        y = step_fn(Mat, y, h)
    return y


In [ ]:
q0 = np.exp(-x**2/50)          # a localized Gaussian wavepacket
T  = 4.0

q_exact = expm(L*T) @ q0        # "exact" reference: matrix exponential (not a solver you should use in Part 2!)

dts = [0.1, 0.05, 0.02, 0.01, 0.005]
err_euler, err_rk4 = [], []
for dt in dts:
    q_e = integrate(euler_step, L, q0, 0, T, dt)
    q_r = integrate(rk4_step,   L, q0, 0, T, dt)
    err_euler.append(np.linalg.norm(q_e - q_exact))
    err_rk4.append(np.linalg.norm(q_r - q_exact))

for dt, ee, er in zip(dts, err_euler, err_rk4):
    print(f'dt={dt:6.3f}   ||error|| Euler = {ee:10.3e}   ||error|| RK4 = {er:10.3e}')

plt.figure(figsize=(5, 4))
plt.loglog(dts, err_euler, 'o-', label='Euler')
plt.loglog(dts, err_rk4, 's-', label='RK4')
plt.xlabel(r'$\Delta t$'); plt.ylabel('error vs. exact (expm)')
plt.legend(); plt.show()


### Q: Try `dt = 0.1` and `dt = 0.05` above (add them to the `dts` list, or rerun with just those). What happens? Which term in $L$ (advection or diffusion) sets the stability limit, and how would you expect that limit to scale with the grid spacing $\Delta x$?

### Q: From the log-log plot, estimate the order of convergence of each method (how does the error scale when $\Delta t$ is halved?). Does it match the nominal order of Euler (1st) and RK4 (4th)?

## Part 2 — Optimal transient energy growth

Among all initial conditions $q(0)=g$ of unit energy $\langle g,g\rangle = g\cdot g = 1$, which one is amplified the most by time $T$? Define the energy growth

$$
G(T) = \frac{\langle q(T), q(T)\rangle}{\langle g, g \rangle} = \frac{q(T)\cdot q(T)}{g \cdot g}.
$$

Finding $g$ that maximises $G(T)$ is a constrained optimisation problem (minimise $\mathcal J = 1/G(T)$ subject to the state equation $\dot q = Lq$, $q(0)=g$). Following exactly the same Lagrangian construction as in Cossu §4.1 (and its extension to space-dependent problems in §6.1), the optimality system is:

$$
\begin{aligned}
\text{state:}\qquad & \dot q = Lq, \quad q(0) = g \\
\text{adjoint:}\qquad & -\dot a = L^{\mathsf T} a, \quad a(T) = -2\, \frac{(g\cdot g)}{(q(T)\cdot q(T))^2}\, q(T) \\
\text{optimality:}\qquad & g = a(0)\, \frac{q(T)\cdot q(T)}{2}
\end{aligned}
$$

Because the discretized state equation is a real linear system, its adjoint operator (for the standard dot-product inner product) is simply the **transpose** $L^{\mathsf T}$ — no derivation needed beyond that.

This system is solved the same way as the iterative methods in Cossu's notes (and, in spirit, like the population update in PS3's PSO): starting from a normalised guess $g$, repeat

1. integrate the state equation **forward** from $t=0$ to $t=T$, starting from $g$, to get $q(T)$;
2. build the terminal adjoint condition $a(T)$ above (with $g\cdot g = 1$);
3. integrate the adjoint equation **backward** from $t=T$ to $t=0$ to get $a(0)$ — note $-\dot a = L^{\mathsf T} a$ means $\dot a = -L^{\mathsf T} a$, so you must integrate with the matrix $-L^{\mathsf T}$;
4. form the new $g$ from the optimality condition and **renormalise it to unit norm** (the problem is scale-invariant, so only the direction of $g$ matters);

until $G(T)$ stops changing. Use your own `rk4_step` (via `integrate`) for both the forward and backward integrations.

In [ ]:
def optimal_growth(L, T, dt, iters=30, tol=1e-8, seed=0):
    N = L.shape[0]
    rng = np.random.default_rng(seed)
    g = rng.standard_normal(N)
    g = g / np.linalg.norm(g)

    negLT = -L.T          # rhs matrix for the adjoint equation da/dt = -L^T a
    G_old = 0.0
    for it in range(iters):
        qT     =                    #>>> integrate the state equation forward, 0 to T, starting from g
        qTqT   = qT @ qT
        aT     =                    #>>> terminal adjoint condition a(T)  (g.g = 1 already)
        a0     =                    #>>> integrate the adjoint equation backward, T to 0, starting from aT
        g_new  =                    #>>> optimality condition
        g      =                    #>>> renormalise g_new to unit norm

        G = qTqT
        if abs(G - G_old) < tol*G:
            break
        G_old = G
    return G, g, it + 1


In [ ]:
dt = 0.01
T  = 4.58
G, g_opt, n_iter = optimal_growth(L, T, dt)

G_exact = np.linalg.svd(expm(L*T), compute_uv=False)[0]**2   # independent check: largest singular value^2 of expm(L*T)

print(f'iterative optimum:  G = {G:.6f}   ({n_iter} outer iterations)')
print(f'exact reference:    G = {G_exact:.6f}   (top singular value of expm(L*T), squared)')
print(f'relative difference: {abs(G - G_exact)/G_exact:.2e}')


In [ ]:
Ts = np.linspace(0.5, 14, 25)
G_iter  = np.array([optimal_growth(L, T, dt)[0] for T in Ts])
G_exact = np.array([np.linalg.svd(expm(L*T), compute_uv=False)[0]**2 for T in Ts])

Topt = Ts[np.argmax(G_iter)]
Gmax = G_iter.max()

plt.figure(figsize=(6, 4))
plt.plot(Ts, G_iter, 'o', label='forward-adjoint iteration')
plt.plot(Ts, G_exact, '-', label='exact (svd of expm(L*T))')
plt.axvline(Topt, color='gray', ls='--', lw=0.8)
plt.xlabel('T'); plt.ylabel('G(T)')
plt.legend(); plt.show()

print(f'optimal horizon T* ~= {Topt:.2f},  G_max ~= {Gmax:.2f}')


In [ ]:
G, g_opt, _ = optimal_growth(L, Topt, dt)
qT_opt = integrate(rk4_step, L, g_opt, 0, Topt, dt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(x, 0, sigma, where=(sigma > 0), color='r', alpha=0.15, label='locally unstable pocket')
ax.plot(x, g_opt / np.max(np.abs(g_opt)), label=r'optimal $q(x,0)$ (normalised)')
ax.plot(x, qT_opt / np.max(np.abs(qT_opt)), label=fr'$q(x,T^*)$, $T^*={Topt:.1f}$ (normalised)')
ax.set_xlabel('x'); ax.legend()
plt.show()


### The full solution $q(x,t)$

The two snapshots above are just the endpoints of an actual solution of the PDE. To see the whole story — the wavepacket travelling through the pocket and being amplified along the way, then decaying once it has left — integrate forward from the optimal $g$ and keep every intermediate state, not just the last one.

In [ ]:
def integrate_traj(step_fn, Mat, y0, t0, t1, dt):
    "Like integrate, but also returns the state q(x,t) at every intermediate time."
    nsteps = int(round(abs(t1 - t0) / dt))
    h = dt if t1 > t0 else -dt
    ts = t0 + h*np.arange(nsteps + 1)
    Q = np.zeros((nsteps + 1, len(y0)))
    Q[0] = y0
    y = y0.copy()
    for n in range(nsteps):
        y =                          #>>> advance one step with step_fn, as in `integrate`
        Q[n+1] = y
    return ts, Q


In [ ]:
ts, Q = integrate_traj(rk4_step, L, g_opt, 0, Topt, dt)
energy = np.sum(Q**2, axis=1)   # ||q(t)||^2 ; energy(0) = 1 since g_opt is unit-norm

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

pcm = ax1.pcolormesh(x, ts, Q, shading='auto', cmap='RdBu_r',
                      vmin=-np.abs(Q).max(), vmax=np.abs(Q).max())
edge = np.sqrt(2*sigma0/sigma2)
ax1.axvline(-edge, color='k', ls='--', lw=0.8)
ax1.axvline( edge, color='k', ls='--', lw=0.8, label='pocket edges')
ax1.set_ylabel('t'); ax1.set_title(r'$q(x,t)$ for the optimal initial condition')
ax1.legend(loc='upper right')
fig.colorbar(pcm, ax=ax1, label='q')

ax2.plot(ts, energy)
ax2.set_xlabel('t'); ax2.set_ylabel(r'$\|q(t)\|^2$')
plt.tight_layout(); plt.show()

print('energy at t=0:', energy[0], ' energy at t=T*:', energy[-1], ' matches G_max found above:', Gmax)


### Q: In the space-time plot of $q(x,t)$, can you see the wavepacket crossing the pocket? Does the running energy $\|q(t)\|^2$ in the lower panel peak at the same $t$ as the $T^*$ found by the $T$-sweep above?

### Q: Where is the optimal initial condition seeded relative to the locally unstable pocket ($\sigma(x)>0$), and where has the response moved to by $t=T^*$? Relate this to the *convective* (as opposed to purely local) nature of the amplification mechanism described as "noise amplifier" behaviour in Bagheri et al. (2009, §II).

### Q: Increase $\sigma_0$ towards the neutral curve (e.g. try $\sigma_0=1.0$, then $\sigma_0=1.1$; check with the eigenvalue plot from Part 0 that the flow is still just barely globally stable) and rerun the $T$-sweep. How does $G_{\max}$ change? What do you expect to happen once $\sigma_0$ is large enough that $L$ has an eigenvalue with positive real part (an "oscillator" rather than a "noise amplifier", in Bagheri et al.'s terminology)?

### Q: How many outer forward/adjoint sweeps did `optimal_growth` need to converge? Compare this to the ~15-20 iterations PSO needed in PS3, or the number of steps steepest descent needed in PS1/PS2. Why might a fixed-point iteration built from a *linear* operator converge so much faster than a general nonlinear optimisation?

### Q: Rerun `optimal_growth` using `euler_step` instead of `rk4_step` for both the forward and adjoint integrations, keeping the same `dt=0.01`. Does it still converge? If you had used the `dt` found to be borderline stable in Part 1 for the *forward* problem, would you expect the *optimisation loop* (forward + adjoint, many outer iterations) to remain stable? Why might it be more, or less, forgiving than a single forward simulation?